# Week 3: Exploratory Data Analysis & Feature Engineering
## Notebook 08: Copulas & Non-Linear Multivariate Dependence

### 1. Conceptual Intuition & Theoretical Background
Standard correlation metrics (e.g. Pearson correlation $r$) only measure **linear** associations. They fail when features exhibit non-linear relationships or asymmetric **tail dependence** (e.g., two assets crash together during market crises but move independently during normal periods).

**Sklar's Theorem** states that any multivariate cumulative distribution function (CDF) $F(x_1, \dots, x_d)$ can be decoupled into its univariate marginal distributions $F_i(x_i)$ and a **Copula** $C$:

$$F(x_1, x_2, \dots, x_d) = C(F_1(x_1), F_2(x_2), \dots, F_d(x_d))$$

Where $u_i = F_i(x_i) \sim \text{Uniform}(0, 1)$ via the **Probability Integral Transform**.

The **Clayton Copula** models extreme lower-tail dependence:
$$C_\theta(u, v) = \max\left( (u^{-\theta} + v^{-\theta} - 1)^{-1/\theta},\ 0 \right), \quad \theta > 0$$
$$\lambda_L = 2^{-1/\theta} \quad (\text{Lower Tail Dependence Coefficient})$$

---

### 2. Scratch Python Implementation: Bivariate Clayton Copula

In [ ]:
import numpy as np

# Scratch Bivariate Copula Generator
def sample_clayton_copula(theta, n_samples=1000):
    """
    Sample (u, v) from Clayton Copula using conditional distribution method:
    v = (u^(-theta) * (w^(-theta / (1 + theta)) - 1) + 1)^(-1 / theta)
    """
    u = np.random.uniform(0.001, 0.999, n_samples)
    w = np.random.uniform(0.001, 0.999, n_samples)
    
    v = (u**(-theta) * (w**(-theta / (1.0 + theta)) - 1.0) + 1.0)**(-1.0 / theta)
    return u, v

# Generate Clayton dependence with parameter theta = 4.0
theta = 4.0
u_samples, v_samples = sample_clayton_copula(theta, n_samples=3000)

# Theoretical lower tail dependence
lambda_L = 2.0 ** (-1.0 / theta)
print(f"Clayton Parameter theta:                {theta}")
print(f"Theoretical Lower Tail Dependence:       {lambda_L:.4f}")

# Inverse transform to desired marginal distributions:
# Feature X ~ Exponential(scale=2.0)
# Feature Y ~ Standard Normal N(0, 1)
from scipy.stats import norm, expon

feature_X = expon.ppf(u_samples, scale=2.0)
feature_Y = norm.ppf(v_samples, loc=0.0, scale=1.0)

# Compute Pearson vs Spearman rank correlation
pearson_r = np.corrcoef(feature_X, feature_Y)[0, 1]
# Empirical rank correlation
rank_u = np.argsort(np.argsort(feature_X))
rank_v = np.argsort(np.argsort(feature_Y))
spearman_r = np.corrcoef(rank_u, rank_v)[0, 1]

print(f"Observed Pearson Linear Correlation:    {pearson_r:.4f}")
print(f"Observed Spearman Rank Correlation:      {spearman_r:.4f}")
print("Notice how rank correlation accurately captures monotonic tail coupling!")

### 3. Step-by-Step Code Walkthrough
1. **`sample_clayton_copula`**: Generates correlated uniform variates $(u, v) \in [0, 1]^2$ using conditional inverse transform sampling.
2. **Probability Integral Transform**: Converts uniform variables into arbitrary marginals (Exponential and Normal) while strictly preserving dependency structure.
3. **Tail Analysis**: Features show asymmetric co-movement during negative shocks.

---

### 4. Industry Applications & Trade-offs
- **Synthetic Tabular Data (CTGAN / CopulaGAN):** Preserves complex multi-modal inter-feature joint distributions.
- **Credit Risk Modeling:** Models simultaneous defaults of loan portfolios during macroeconomic downturns.

---

### 5. Top Interview Questions & Answers
**Q1: Why is Pearson correlation coefficient insufficient for modeling credit default risk?**
> **Answer:** Pearson correlation only evaluates linear relationships and assumes elliptical distributions. In defaults, extreme events show non-linear tail clustering (asymmetric co-crashes) that only copulas capture.

**Q2: What is the Probability Integral Transform (PIT)?**
> **Answer:** If $X$ is a continuous random variable with CDF $F_X$, then $U = F_X(X)$ is uniformly distributed on $[0, 1]$. Its inverse $F_X^{-1}(U)$ transforms uniform noise back into the original distribution.